In [ ]:
%matplotlib inline

# Generative quantum advantage for classical and quantum problems

Generative machine learning is all about inferring and sampling from
probability distributions, and sampling output distributions of quantum
computers is known to be classically hard. So proving quantum advantages
for generative quantum machine learning is easy, right? Unfortunately,
things are not so simple. As H. Huang and colleagues point out in their
recent preprint *"Generative quantum advantage for classical and quantum
problems"*, claiming an advantage for generative machine learning should
not only require the ability to sample from a hard distribution, but
also to be able to learn it efficiently from data, and they investigate
a specific scenario in which this is possible.

In this demo we will unpack one of the main results of their paper to
understand its core mechanics. We will see that the problem is
constructed so that learning the hard distribution boils down to
performing single-qubit tomography, and we will discuss the scope of
this technique in relation to practical AI. In particular, we will focus
on the first theorem (Theorem 1) of the paper, since it aligns closest
with the notion of generative machine learning in the classical
literature. It is informally stated as:

> Theorem 1 (Informal: Classically hard, quantumly easy generative
> models)
>
> Under standard complexity-theoretic conjectures, there exist
> distributions $p(y|x)$ mapping classical n-bit strings to m-bit
> strings that a quantum computer can efficiently learn to generate
> using classical data samples, but are hard to generate with classical
> computers.

To show the above, we need to do a couple of things:

-   Identify a classically 'hard' conditional distribution $p(y|x)$ that
    corresponds to a family of quantum circuits. For this, we can
    leverage some existing results about the hardness of sampling.
-   Show that, with access to a dataset obtained by querying and
    sampling from $p(y|x)$, we can infer the circuits that produced the
    data, and can therefore generate more data.

The paper gives a couple of circuit structures that can be used. We will
focus on the simplest, which they term instantaneously deep quantum
neural networks (IDQNNs).


# Instantaneously deep quantum neural networks

Instantaneously deep quantum neural networks, or IDQNNs, are a
particular type of shallow parameterized quantum circuits. The qubits of
the circuit live on a lattice, which we'll take to be a 2D lattice, and
index the qubits by their lattice positions $(i,j)$. To sample from the
circuit one does the following.

## Recipe for sampling from an IDQNN

1.  Prepare each qubit on the lattice in the $\ket{+}$ state.
2.  Entangle the qubits by performing controlled-Z gates between some
    pairs of nearest-neighbour qubits on the lattice. If the qubits are
    horizontal neighbours on the lattice, a CZ is always applied.
3.  Perform a single-qubit Z rotation
    $U_{z}(\theta_{ij})=\exp(-\frac{i}{2}\theta_{ij}Z)$ with parameter
    $\theta_{ij}$ on each of the qubits
4.  Measure all qubits in the X basis to produce outcomes $y_{ij}$

We can depict this graphically as follows.

![](https://github.com/osbama/KBM608/blob/main/hands-on/hands-on-8-images/pennylane-demo-generative-quantum-advantage-fig1.png?raw=1)

Note that this is not a circuit diagram, but a graphical representation
of the circuit description above: qubits are denoted by black dots, CZ
gates are lines between dots, the angles specify the single-qubit
rotations, and the blue $y_{ij}$ are the X-measurement outcomes. We'll
also use the vector $\boldsymbol{y}$ from now on to denote all the
$y_{ij}$.

The above corresponds to a circuit acting on a 2D lattice of 12 qubits,
and this structure corresponds to a shallow circuit since the circuit
depth does not depend on the size of the lattice. However, we can map
this onto an equivalent deep 1D circuit with only 3 qubits by viewing
the circuit as a measurement based quantum computation (MBQC) recipe.
When viewed from this perspective, the horizontal dimension of the
lattice becomes a time axis, so that at time 0, the system consists of
just three qubits (the first vertical axis) prepared in the $\ket{+}$
state. After applying all CZ and rotation gates that involve these
qubits and measuring them in the X basis, the state is teleported to the
next line of three qubits (the precise state will depend on the
measurement outcomes $y_{11}$, $y_{12}$, $y_{13}$). Repeating this
process until we arrive at the end of the lattice therefore defines a
type of stochastic quantum circuit acting on three qubits. The precise
way to make this mapping is well known from MBQC theory, and is a bit
tricky (see Appendix H2 of their paper[^1]), so we won't bore you with
the details here.

If you apply the mapping to our example IDQNN, you find the following
circuit:

![](https://github.com/osbama/KBM608/blob/main/hands-on/hands-on-8-images/pennylane-demo-generative-quantum-advantage-fig2.png?raw=1)

The circuit structure for layers 2 and 3 is the same as for layer 1,
where the CZ structure is determined by the vertically acting CZ gates
that appear in the second and third vertical axis of the lattice. The
inputs to the Z gates are classical controls, i.e. the gate is applied
only if $y_{ij}=1$.

To generate samples from this deep circuit we do the following:

## Recipe for sampling from the deep circuit

1.  Generate all bits $y_{ij}$ for $i<4$ uniformly at random using a
    classical random number generator.
2.  Run the above circuit, controlling the Z gates on these bits.
3.  Measure the output of the circuit to obtain the final three bits
    $y_{41}$, $y_{42}$, $y_{43}$.

One can show that the distribution $p(\boldsymbol{y})$ obtained with the
above recipe is identical to the one we described for the IDQNN, and so
the two methods are indistinguishable if just given samples
$\boldsymbol{y}$.

If these two circuits lead to the same distribution then why did we do
this? The reason is that qubit counts on quantum hardware are still
limited. By implementing the deep 1D circuit on a few qubits however,
you can simulate the distribution of a 2D shallow circuit on many more
qubits (the authors call this circuit \'compression\'). This trick is
used to simulate a shallow IDQNN circuit on 816 qubits using a deep
circuit with just 68 qubits. To do this, they actually work with a deep
circuit on a 2D lattice, and map it to a shallow circuit on a 3D
lattice. This obviously complicates things a bit (and makes drawing
pictures a lot harder!) so we will stick to the 2D vs 1D example above;
in the end, it will contain everything we need to understand the result
for higher dimensional lattices.

# Proving hardness for sampling

It turns out that---if we remove the classically controlled Z gates for
now---the circuit structure of the deep circuit above is universal. That
is, any $n$-qubit circuit with two qubit gates can be efficiently
approximated by sequential layers of Hadamards, Z rotations and
controlled-Z gates on an $n$ qubit computational basis input. We can
therefore use this fact to define a circuit that is hard to sample from
classically: simply take your favourite pre-existing hardness results
for sampling (for example,[^2]) and compile the circuit to the H, RZ, CZ
gateset. We can then embed this into the precise structure we had above
by inserting the classically controlled-Z gates at every layer. If we
happen to sample the all-zero bitstring for the $y_{ij}$ values that
control these gates, then we will sample from this hard distribution. In
this sense the distribution $p(\boldsymbol{y})$ is 'hard' since any
classical algorithm will fail to reproduce the full statistics in this
case. Moreover, since the distribution of the IDQNN is identical, it
follows that the corresponding IDQNN is also hard to sample from.

# Adding inputs states

At this point, we have a shallow circuit called an IDQNN, a way to map
it to a deep circuit structure, and an argument that the distributions
$p(\boldsymbol{y})$ resulting from these circuits are hard to sample
from classically. However, we don't yet have everything in order to be
able to learn. The last ingredient we need comes in the form of an input
$x$. This will mean that rather than working with the the probability
distribution $p(\boldsymbol{y})$, we will work with a conditional
probability distribution $p(\boldsymbol{y}|x)$.

For each $x$, the probability distribution $p(\boldsymbol{y}|x)$
corresponds to a IDQNN where---rather than all qubits being in the
$\vert + \rangle$ state---each input qubit can be prepared in either the
$\vert + \rangle$ or $\vert 0 \rangle$ state, which is determined by the
input $x$. We therefore adapt the first step of our recipe for the
IDQNN:

## Recipe for sampling from an IDQNN with inputs

1.  Prepare each qubit in either the $\vert + \rangle$ or
    $\vert 0 \rangle$ state, depending on $x$.
2.  Perform steps 2-4 as before.

In order to be able to prove the result, the choice and distribution of
possible input states must satisfy a particular property called 'local
decoupling' (see Appendix C2 of their paper[^3]). One particularly
simple choice that will work for our 2D IDQNN is the following choice of
three inputs, $x=0,1,2$ (in the paper a different choice is used, but
the result will be the same).

If $x=0$, all input qubits are prepared in the $\vert + \rangle$ state.
If $x=1$, all qubits on the 'even diagonals' of the lattice are prepared
in $\vert + \rangle$, the remaining are prepared in $\vert 0 \rangle$.
If $x=2$, all qubits on the 'odd diagonals' of the lattice are prepared
in $\vert + \rangle$, the remaining are prepared in $\vert 0 \rangle$.

Pictorially, the choice looks like this.

![](https://github.com/osbama/KBM608/blob/main/hands-on/hands-on-8-images/pennylane-demo-generative-quantum-advantage-fig3.png?raw=1)

If the input is 0, we already know what happens; this is just the IDQNN
described in the previous sections. If the input is 1 or 2, things get
very simple. Note that the CZ gate is symmetric, and can be written as

$$CZ = \vert 0 \rangle\langle 0 \vert \otimes \mathbb{I} +  \vert 1 \rangle\langle 1 \vert \otimes Z =  \mathbb{I} \otimes  \vert 0 \rangle\langle 0 \vert + Z \otimes  \vert 1 \rangle\langle 1 \vert.$$

Thus, if a CZ gate acts on the state $\vert 0 \rangle$ on either side,
the effect is that it becomes the identity. Since every CZ gate hits a
qubit in the $\vert 0 \rangle$ state for these inputs, we can actually
remove them all. For example, the input $x=1$ actually just corresponds
to an unentangled product state:

![](https://github.com/osbama/KBM608/blob/main/hands-on/hands-on-8-images/pennylane-demo-generative-quantum-advantage-fig4.png?raw=1)

By performing mid-circuit measurements and resetting qubits, we can
easily reproduce the statistics for the inputs $x=1,2$. For example, for
$x=1$, the circuit looks as follows (we have removed the rotation gates
for the qubits prepared in $\vert 0 \rangle$ since this results in a
global phase only).

![](https://github.com/osbama/KBM608/blob/main/hands-on/hands-on-8-images/pennylane-demo-generative-quantum-advantage-fig5.png?raw=1)

The authors argue that the conditional distribution
$p(\boldsymbol{y}|x)$ should also be considered hard to sample from
classically, since the input $x=0$ corresponds to the case of the \'no
input\' IDQNN of the previous sections, for which we have already argued
hardness. For the inputs $x=1$ and $x=2$, however, the resulting
distribution has an efficient classical simulation since it corresponds
to measurements made on unentangled single qubits.

# The learning problem

We now have a classically hard conditional distribution
$p(\boldsymbol{y}|x)$, where each input $x$ corresponds to a IDQNN with
inputs that we know how to simulate with a deeper circuit on fewer
qubits. At this point we are ready to learn.

We first need a dataset, which we create by repeating the following $N$
times

-   Randomly sample an input $x=0,1,2$.
-   Implement the deep circuit that simulates the IDQNN for this input
    to generate a set of outcomes $\boldsymbol{y}$. Add the pair
    $(x,\boldsymbol{y})$to the dataset.

The precise definition of learning is given by definition 13 in the
Appendix of the paper:

> Definition 13 (The task of learning to generate classical bitstrings)
>
> We are given a dataset of input-output bitstring pairs
> $(x,\boldsymbol{y})$. Each output bitstring $\boldsymbol{y}$ is
> sampled according to an unknown conditional distribution
> $p(\boldsymbol{y}|x)$. The goal is to learn a model from the dataset
> that can generate new output bitstrings $\boldsymbol{y}$ according to
> the unknown distribution $p(\boldsymbol{y}|x)$ for any given new input
> bitstring $x$.

Although the above definition suggests the conditional distribution is
unknown, we actually know a lot about it. In particular, we need to work
under the assumption that we know the exact structure of the quantum
circuits that produce the data, except for the rotation angles
$\theta_{ij}$ (i.e. this is included in the \`prior knowledge' of the
problem). To learn, we therefore just need to infer the parameters
$\theta_{ij}$ from the data, which will allow us to generate new data by
simply implementing the resulting circuits. This is very different from
real world machine learning problems, where such a precise parametric
form of the ground truth distribution is rarely known.

So how do we infer the parameters $\theta_{ij}$ from data? Consider for
example the data for input $x=1$, and the outcome $y_{12}$. From the
above circuit we see that in this case the outcome is produced by this
single qubit circuit:

![](https://github.com/osbama/KBM608/blob/main/hands-on/hands-on-8-images/pennylane-demo-generative-quantum-advantage-fig6.png?raw=1)

This is a measurement on a rotated single-qubit state, for which the
expectation value for $y_{12}$ is

$$\langle y_{12} \rangle = (1-\cos(\theta_{12}))/2.$$

Rearranging this equation we have

$$\theta_{12} = \frac{1}{2} \arccos(1 - 2\langle y_{12} \rangle).$$

All we have to do to infer $\theta_{12}$ is to look at the data for
$x=1$, estimate the expectation $\langle y_{12} \rangle$ from the
corresponding $y_{12}$ values, and use the above formula; no gradients
or training required (so clearly no barren plateaus either)! Note that
all we are doing here is a form of single-qubit tomography, which you
might encounter in a first course of quantum information.

The remaining parameters can be estimated in a similar way depending on
whether they live on the even diagonal (which requires $x=1$) or the odd
diagonal (which requires $x=2$). From the Hoeffding inequality, we can
be sure that the estimates are close to the true values with high
probability, and we thus have learned the parameters to low error. With
this knowledge we can now sample data for the $x=0$ input and this is
known to be classically hard, and in this sense we have learned the
distribution.

# Does this bring us closer to useful quantum AI?

Now that the technicalities are out of the way, we can ask the really
important question: does this bring us closer to genuinely useful QML
algorithms? This question is speculated on briefly in the conclusion of
the paper where it is mentioned that \"the precise remaining steps
towards useful generative quantum advantage remain an open question\".
But why is usefulness so enigmatic? As we explain below, a large part of
the reason is due to the fact that the setup we considered is
significantly different to that of real-world generative machine
learning problems.

## The necessity of prior knowledge

The first of these differences concerns the 'prior knowledge' that needs
to be assumed in order to prove the result. Notice that, if we were
given the data ${(x, \boldsymbol{y})}$ but told nothing else, we would
not have known how to infer the parameters, nor how to produce new
samples once these parameters were known, since these both required
knowledge of the circuit. That is, precise knowledge of the circuit
structures that produced the data was necessary in order to learn. In
reality, such a precise form of the ground truth distribution is not
known and models have to be learnt with vastly less prior knowledge. In
their current form, these techniques therefore only appear useful for
tasks related to quantum circuit tomography, where such knowledge is
part of the problem description.

## Learning from simple statistics

The second difference concerns how the parameters were learned. The
parameters of modern classical generative models must be learned by an
iterative process (often gradient descent) that extracts their values
from complex, multidimensional correlations that are present in the
statistics of the training data. For example, the parameters of
convolutional filters in image models are adjusted to capture highly
non-linear correlations, such as how edges, textures, and object parts
co-occur across different spatial locations. In the quantum setup,
although the distribution for the input $x=0$ is undoubtedly complex,
the data that is used for learning comes from the much simpler product
distributions in which there are no correlations between bits.

As an illustrative comparison, imagine a large classical generative
model (such as a diffusion model or a transformer) with parameters
$\theta$ and corresponding output distribution $p(\boldsymbol{y})$.
Suppose we want to learn the parameters $\theta$ from data. To do this
we construct a conditional distribution $p(\boldsymbol{y}|x)$ which does
the following:

-   For $x=0$, the model samples the generative model distribution
    $p(\boldsymbol{y})$
-   For $x=1$, $\boldsymbol{y}$ just returns the parameters $\theta$

Obviously, we can learn the parameters of the model from
$p(\boldsymbol{y}|x)$: we just look at the data for $x=1$ and read them
off directly from $\boldsymbol{y}$. Our quantum example is not
dramatically different from this, since for the inputs $x=1,2$ we have a
simple method to read off the parameters from the statistics
(single-qubit tomography), and this method is known beforehand rather
than being learned from the data. In effect, we have set up the problem
so that inferring parameters is straightforward for some inputs, whilst
sampling is hard for others, and this process of learning is very
different from the complex process that occurs in modern neural
networks. We note that the specific example in the paper is more
involved than this, and uses a higher dimensional lattice and a
different set of inputs, but the strategy is the same: for each
parameter, there is a reasonable fraction of the inputs that leaves the
relevant qubit unentangled from the rest, and single-qubit statistics
reveals the desired value.

## What can lead us to genuine usefulness?

In order to uncover genuine usefulness in quantum machine learning we
therefore need to move to scenarios that mirror the assumptions of
realistic learning problems. If the flipside of this is that proving
complexity theoretic separations becomes seemingly impossible, then
perhaps they are not the right goals to be pursuing[^4]? A more fruitful
direction may be to instead look for other tools to evaluate the
performance of quantum models. Although this means waving goodbye to
computational complexity theory, this freedom just might be the fresh
perspective we need to uncover genuinely useful quantum machine learning
algorithms.

[^1]: H. Huang, M. Broughton, N. Eassa, H. Neven, R. Babbush, J. R.
    McClean \"Generative quantum advantage for classical and quantum
    problems.\" [arXiv:2509.09033](https://arxiv.org/abs/2509.09033),
    2025.

[^2]: T. Bergamaschi; C. Chen; Y. Liu \"Quantum Computational Advantage
    with Constant-Temperature Gibbs Sampling.\" [2024 IEEE 65th Annual
    Symposium on Foundations of Computer Science
    (FOCS)](https://ieeexplore.ieee.org/document/10756075), 2024.

[^3]: H. Huang, M. Broughton, N. Eassa, H. Neven, R. Babbush, J. R.
    McClean \"Generative quantum advantage for classical and quantum
    problems.\" [arXiv:2509.09033](https://arxiv.org/abs/2509.09033),
    2025.

[^4]: M. Schuld, N. Killoran \"Is Quantum Advantage the Right Goal for
    Quantum Machine Learning?.\" [PRX Quantum 3,
    030101](https://journals.aps.org/prxquantum/abstract/10.1103/PRXQuantum.3.030101),
    2022.


# Quantum advantage in learning from experiments

This demo is based on the article [Quantum advantage in learning from
experiments](https://arxiv.org/abs/2112.00778) [\[1\]](#ref1) by
Hsin-Yuan Huang and co-authors. The article investigates the following
question:

*How useful is access to quantum memory for quantum machine learning?*

They show that access to quantum memory can make a big difference, and
prove that there exist learning problems for which algorithms with
quantum memory require *exponentially less resources* than those
without. We look at one learning task studied in [\[1\]](#ref1) for
which this is the case.

## The learning task

The learning task we focus on involves deciding if a unitary is
time-reversal symmetric (we'll call them T-symmetric) or not.
Mathematically, time-reversal symmetry in quantum mechanics involves
reversing the sense of $i$ so that $i \rightarrow -i.$ Hence, a unitary
$U$ is T-symmetric if

$$U^*=U.$$

Now for the learning task. Let's say we have a bunch of quantum circuits
$U_1, \cdots, U_n,$ some of which are T-symmetric and some not, but we
are not told which ones are which.


![](https://github.com/osbama/KBM608/blob/main/hands-on/hands-on-8-images/fig1b.png?raw=1)


The task is to design an algorithm to determine which of the $U$'s are
T-symmetric and which are not, given query access to the unitaries. Note
that we do not have any labels here, so this is an unsupervised learning
task. To make things concrete, let's consider unitaries acting on 8
qubits. We will also limit the number of times we can use each unitary:


In [ ]:
qubits = 8  # the number of qubits on which the unitaries act
n_shots = 100  # the number of times we can use each unitary

# Experiments with and without a quantum memory


To tackle this task we consider experiments with and without quantum
memory. We also assume that we have access to a single physical
realization of each unitary; in other words, we do not have multiple
copies of the devices that implement $U_i.$

An experiment without quantum memory can therefore only make use of a
single query to $U_i$ in each circuit, since querying $U_i$ again would
require storing the state of the first query in memory and re-using the
unitary. In the paper these experiments are called **conventional
experiments**.

Experiments with quantum memory do not have the limitations of
conventional experiments. This means that multiple queries can be made
to $U_i$ in a single circuit, which can be realized in practice by using
a quantum memory. These experiments are called **quantum-enhanced
experiments**.

Note that we are not comparing classical and quantum algorithms here,
but rather two classes of quantum algorithms.


![](https://github.com/osbama/KBM608/blob/main/hands-on/hands-on-8-images/experiments.png?raw=1)


# The conventional way


First, we will try to solve the task with a conventional experiment. Our
strategy will be as follows:

-   For each $U_i,$ we prepare `n_shots` copies of the state
    $U_i\vert0\rangle$ and measure each state to generate classical
    measurement data.
-   Use an unsupervised classical machine learning algorithm (kernel
    PCA), to try and separate the data into two clusters corresponding
    to T-symmetric unitaries vs. the rest.

If we succeed in clustering the data then we have successfully managed
to discriminate the two classes!


![](https://github.com/osbama/KBM608/blob/main/hands-on/hands-on-8-images/fig2b.png?raw=1)


To generate the measurement data, we will measure the states
$U_i\vert0\rangle$ in the $y$ basis. The local expectation values take
the form

$$E_i  = \langle 0\vert U^{\dagger}\sigma_y^{(i)} U \vert 0 \rangle.$$

where $\sigma_y^{(i)}$ acts on the $i^{\text{th}}$ qubit.

Using the fact that $\sigma_y^*=-\sigma_y$ and the property $U^*=U$ for
T-symmetric unitaries, one finds

$$E_i^*=\langle 0\vert (U^{\dagger})^*(\sigma_y^{(i)})^* (U)^* \vert 0 \rangle = - \langle 0\vert U^{\dagger}\sigma_y^{(i)} U \vert 0 \rangle = - E_i.$$

Since $E_i$ is a real number, the only solution to this is $E_i=0,$
which implies that all local expectations values are 0 for this class.

For general unitaries it is not the case that $E_i=0,$ and so it seems
as though this will allow us to discriminate the two classes of circuits
easily. However, for general random unitaries the local expectation
values approach zero exponentially with the number of qubits: from
finite measurement data it can still be very hard to see any difference!
In fact, in the article [exponential separations between learning with
and without quantum memory](https://arxiv.org/abs/2111.05881)
[\[2\]](#ref2) it is proven that using conventional experiments, any
successful algorithm *must* use the unitaries an exponential number of
times.


Let's see how this looks in practice. First we define a function to
generate random unitaries, making use of Pennylane's
[RandomLayers](https://pennylane.readthedocs.io/en/stable/code/api/pennylane.RandomLayers.html)
template. For the time-symmetric case we will only allow for Y
rotations, since these unitaries contain only real numbers, and
therefore result in T-symmetric unitaries. For the other unitaries, we
will allow rotations about X,Y, and Z.


In [ ]:
import pennylane as qml
from pennylane.templates.layers import RandomLayers
import numpy as np

np.random.seed(234087)

layers, gates = 10, 10  # the number of layers and gates used in RandomLayers


def generate_circuit(shots):
    """
    generate a random circuit that returns a number of measuement samples
    given by shots
    """
    dev = qml.device("lightning.qubit", wires=qubits)

    @qml.set_shots(shots)
    @qml.qnode(dev)
    def circuit(ts=False):

        if ts == True:
            ops = [qml.RY]  # time-symmetric unitaries
        else:
            ops = [qml.RX, qml.RY, qml.RZ]  # general unitaries

        weights = np.random.rand(layers, gates) * np.pi
        RandomLayers(weights, wires=range(qubits), rotations=ops, seed=np.random.randint(0, 10000))

        return [qml.sample(op=qml.PauliY(q)) for q in range(qubits)]

    return circuit

let's check if that worked:


In [ ]:
# the measurement outcomes for the first 3 shots
circuit = generate_circuit(n_shots)
print(np.array(circuit(ts=True))[:, 0:3])
print("\n")
print(np.array(circuit(ts=False))[:, 0:3])

Now we can generate some data. The first 30 circuits in the data set are
T-symmetric and the second 30 circuits are not. Since we are in an
unsupervised setting, the algorithm will not know this information.


In [ ]:
circuits = 30  # the number of circuits in each data set

raw_data = []

for ts in [True, False]:
    for __ in range(circuits):
        circuit = generate_circuit(n_shots)
        raw_data.append(circuit(ts=ts))

Before feeding the data to a clustering algorithm, we will process it a
little. For each circuit, we calculate the mean and the variance of each
output bit and store this in a vector of size `2*qubits`. These vectors
make up our classical data set.


In [ ]:
def process_data(raw_data):
    "convert raw data to vectors of means and variances of each qubit"

    raw_data = np.array(raw_data)
    nc = len(raw_data)  # the number of circuits used to generate the data
    nq = len(raw_data[0])  # the number of qubits in each circuit
    new_data = np.zeros([nc, 2 * nq])

    for k, outcomes in enumerate(raw_data):
        means = [np.mean(outcomes[q, :]) for q in range(nq)]
        variances = [np.var(outcomes[q, :]) for q in range(nq)]
        new_data[k] = np.array(means + variances)

    return new_data


data = process_data(raw_data)

Now we use scikit-learn's [kernel
PCA](https://en.wikipedia.org/wiki/Kernel_principal_component_analysis)
package to try and cluster the data. This performs principal component
analysis in a high dimensional feature space defined by a kernel (below
we use the radial basis function kernel).


In [ ]:
from sklearn.decomposition import KernelPCA
from sklearn import preprocessing

kernel_pca = KernelPCA(
    n_components=None, kernel="rbf", gamma=None, fit_inverse_transform=True, alpha=0.1
)

# rescale the data so it has unit standard deviation and zero mean.
scaler = preprocessing.StandardScaler().fit(data)
data = scaler.transform(data)
# try to cluster the data
fit = kernel_pca.fit(data).transform(data)

Let's plot the result. Here we look at the first two principal
components.


In [ ]:
import matplotlib.pyplot as plt

# make a colour map for the points
c = np.array([0 for __ in range(circuits)] + [1 for __ in range(circuits)])

plt.scatter(fit[:, 0], fit[:, 1], c=c)
plt.show()

Looks like the algorithm failed to cluster the data. We can try to get a
separation by increasing the number of shots. Let's increase the number
of shots by 100 and see what happens.


In [ ]:
n_shots = 10000  # 100 x more shots

raw_data = []

for ts in [True, False]:
    for __ in range(circuits):
        circuit = generate_circuit(n_shots)
        raw_data.append(circuit(ts=ts))

data = process_data(raw_data)
scaler = preprocessing.StandardScaler().fit(data)
data = scaler.transform(data)

fit = kernel_pca.fit(data).transform(data)

plt.scatter(fit[:, 0], fit[:, 1], c=c)
plt.show()

Now we have a separation, however we required a lot of shots from the
quantum circuit. As we increase the number of qubits, the number of
shots we need will scale exponentially (as shown in [\[2\]](#ref2)), and
so conventional strategies cannot learn to separate the data
efficiently.


# The quantum-enhanced way

Now let's see what difference having a quantum memory can make. Instead
of using a single unitary to generate measurement data, we will make use
of twice the number of qubits, and apply the unitary twice:


![](https://github.com/osbama/KBM608/blob/main/hands-on/hands-on-8-images/fig3b.png?raw=1)


In practice, this could be done by storing the output state from the
first unitary in quantum memory and preparing the same state by using
the unitary again. Let's define a function `enhanced_circuit()` to
implement that. Note that since we now have twice as many qubits, we use
half the number of shots as before so that the total number of uses of
the unitary is unchanged.


In [ ]:
n_shots = 50
qubits = 8

dev = qml.device("lightning.qubit", wires=qubits * 2)


def CNOT_sequence(control_wires, target_wires):
    """Apply CNOTs in sequence using the provided control and target wires"""
    for c_wire, t_wire in zip(control_wires, target_wires):
        qml.CNOT([c_wire, t_wire])


@qml.set_shots(n_shots)
@qml.qnode(dev)
def enhanced_circuit(ts=False):
    "implement the enhanced circuit, using a random unitary"

    if ts == True:
        ops = [qml.RY]
    else:
        ops = [qml.RX, qml.RY, qml.RZ]

    weights = np.random.rand(layers, n_shots) * np.pi
    seed = np.random.randint(0, 10000)

    for q in range(qubits):
        qml.Hadamard(wires=q)

    CNOT_sequence(control_wires=range(qubits), target_wires=range(qubits, 2 * qubits))
    RandomLayers(weights, wires=range(0, qubits), rotations=ops, seed=seed)
    RandomLayers(weights, wires=range(qubits, 2 * qubits), rotations=ops, seed=seed)
    CNOT_sequence(control_wires=range(qubits), target_wires=range(qubits, 2 * qubits))

    for q in range(qubits):
        qml.Hadamard(wires=q)

    return [qml.sample(op=qml.PauliZ(q)) for q in range(2 * qubits)]

Now we generate some raw measurement data, and calculate the mean and
variance of each qubit as before. Our data vectors are now twice as long
since we have twice the number of qubits.


In [ ]:
raw_data = []

for ts in [True, False]:
    for __ in range(circuits):
        raw_data.append(enhanced_circuit(ts))

data = process_data(raw_data)

Let's throw that into Kernel PCA again and plot the result.


In [ ]:
kernel_pca = KernelPCA(
    n_components=None, kernel="rbf", gamma=None, fit_inverse_transform=True, alpha=0.1
)

scaler = preprocessing.StandardScaler().fit(data)
data = scaler.transform(data)

fit = kernel_pca.fit(data).transform(data)

c = np.array([0 for __ in range(circuits)] + [1 for __ in range(circuits)])
plt.scatter(fit[:, 0], fit[:, 1], c=c)
plt.show()

Kernel PCA has perfectly separated the two classes! In fact, all the
T-symmetric unitaries have been mapped to the same point. This is
because the circuit is actually equivalent to performing
$U^TU\otimes \mathbb{I}\vert 0 \rangle,$ which for T-symmetric unitaries
is just the identity operation.

To see this, note that the Hadamard and CNOT gates before
$U_i\otimes U_i$ map the $\vert0\rangle$ state to the maximally entanged
state
$\vert \Phi^+\rangle = \frac{1}{\sqrt{2}}(\vert 00...0\rangle+ \vert11...1\rangle,$
and the gates after $U_i\otimes U_i$ are just the inverse
transformation. The probability that all measurement outcomes give the
result $+1$ is therefore.

$$p(11\cdots 1) = \langle \Phi^+ \vert U_i \otimes U_i \vert\Phi^+ \rangle.$$

A well known fact about the maximally entanged state is that
$U\otimes \mathbb{I}\vert\Phi^+\rangle= \mathbb{I}\otimes U^T\vert\Phi^+\rangle.$
The probabilty is therefore

$$p(11\cdots 1) = \langle \Phi^+ \vert U_i^T U_i \otimes \mathbb{I} \vert\Phi^+ \rangle.$$

For T-symmetric unitaries $U_i^T=U_i^\dagger,$ so this probability is
equal to one: the $11\cdots 1$ outcome is always obtained.

If we look at the raw measurement data for the T-symmetric unitaries:


In [ ]:
np.array(raw_data[0])[:, 0:5]  # outcomes of first 5 shots of the first T-symmetric circuit

We see that indeed this is the only measurement outcome.

To make things a bit more interesting, let's add some noise to the
circuit. We will define a function `noise_layer(epsilon)` that adds some
random single qubit rotations, where the maximum rotation angle is
`epsilon`.


In [ ]:
def noise_layer(epsilon):
    "apply a random rotation to each qubit"
    for q in range(2 * qubits):
        angles = (2 * np.random.rand(3) - 1) * epsilon
        qml.Rot(angles[0], angles[1], angles[2], wires=q)

We redefine our `enhanced_circuit()` function with a noise layer applied
after the unitaries


In [ ]:
@qml.set_shots(n_shots)
@qml.qnode(dev)
def enhanced_circuit(ts=False):
    "implement the enhanced circuit, using a random unitary with a noise layer"

    if ts == True:
        ops = [qml.RY]
    else:
        ops = [qml.RX, qml.RY, qml.RZ]

    weights = np.random.rand(layers, n_shots) * np.pi
    seed = np.random.randint(0, 10000)

    for q in range(qubits):
        qml.Hadamard(wires=q)

    CNOT_sequence(control_wires=range(qubits), target_wires=range(qubits, 2 * qubits))
    RandomLayers(weights, wires=range(0, qubits), rotations=ops, seed=seed)
    RandomLayers(weights, wires=range(qubits, 2 * qubits), rotations=ops, seed=seed)
    noise_layer(np.pi / 4)  # added noise layer
    CNOT_sequence(control_wires=range(qubits, 2 * qubits), target_wires=range(qubits))

    for q in range(qubits):
        qml.Hadamard(wires=qubits + q)

    return [qml.sample(op=qml.PauliZ(q)) for q in range(2 * qubits)]

Now we generate the data and feed it to kernel PCA again.


In [ ]:
raw_data = []

for ts in [True, False]:
    for __ in range(circuits):
        raw_data.append(enhanced_circuit(ts))

data = process_data(raw_data)

kernel_pca = KernelPCA(
    n_components=None, kernel="rbf", gamma=None, fit_inverse_transform=True, alpha=0.1
)
scaler = preprocessing.StandardScaler().fit(data)
data = scaler.transform(data)
fit = kernel_pca.fit(data).transform(data)

c = np.array([0 for __ in range(circuits)] + [1 for __ in range(circuits)])
plt.scatter(fit[:, 0], fit[:, 1], c=c)
plt.show()

Nice! Even in the presence of noise we still have a clean separation of
the two classes. This shows that using entanglement can make a big
difference to learning.


\[1\] *Quantum advantage in learning from experiments*, Hsin-Yuan Huang
et. al., [arxiv:2112.00778](https://arxiv.org/pdf/2112.00778.pdf) (2021)

\[2\] *Exponential separations between learning with and without quantum
memory*, Sitan Chen, Jordan Cotler, Hsin-Yuan Huang, Jerry Li,
[arxiv:2111.05881](https://arxiv.org/abs/2111.05881) (2021)
